In [2]:
from google.colab import files

uploaded = files.upload()

Saving energy_master.csv to energy_master.csv


In [6]:
#energy balance dataframe:
import pandas as pd

# Load the energy_master.csv file into a DataFrame named 'energy'
energy = pd.read_csv('energy_master.csv')

# Ensure 'utc_timestamp' is a datetime object and set it as index if needed for 'energy.index'
# Assuming 'utc_timestamp' is the desired index column based on common usage in energy data.
if 'utc_timestamp' in energy.columns:
    energy['utc_timestamp'] = pd.to_datetime(energy['utc_timestamp'])
    energy = energy.set_index('utc_timestamp')

system = pd.DataFrame(index=energy.index)

system['load'] = energy['load']
system['wind'] = energy['wind']
system['solar'] = energy['solar']

system['renewables'] = (
    system['wind'] +
    system['solar']
)

system['net_balance'] = (
    system['renewables'] -
    system['load']
)

system.head()

,load,wind,solar,renewables,net_balance
utc_timestamp,,,,,
2018-10-01 05:00:00+00:00,50370.0,10491.0,208.0,10699.0,-39671.0
2018-10-01 06:00:00+00:00,52844.0,10705.0,2008.0,12713.0,-40131.0
2018-10-01 07:00:00+00:00,53412.0,10201.0,5132.0,15333.0,-38079.0
2018-10-01 08:00:00+00:00,54296.0,10670.0,8344.0,19014.0,-35282.0
2018-10-01 09:00:00+00:00,55552.0,12937.0,10209.0,23146.0,-32406.0


Το σύστημα έχει μόνιμο ενεργειακό έλλειμμα (deficit).

Αυτό είναι απολύτως ρεαλιστικό για ένα ηλεκτρικό σύστημα όπου:

η ζήτηση είναι πολύ μεγαλύτερη από την παραγωγή από ΑΠΕ,
και άρα χρειάζεται είτε:
αποθήκευση,
είτε συμβατική παραγωγή,
είτε εισαγωγές από το δίκτυο

In [7]:
#Battery Simulator
battery_capacity = 100000      # MWh
initial_soc = 50000            # αρχική φόρτιση
charge_efficiency = 0.95
discharge_efficiency = 0.95

max_charge = 10000
max_discharge = 10000

In [8]:
#Simulation
system['battery_soc'] = 0.0
system['battery_flow'] = 0.0
system['grid_import'] = 0.0
system['curtailment'] = 0.0

soc = initial_soc

for i in range(len(system)):

    balance = system.iloc[i]['net_balance']

    # surplus renewables
    if balance > 0:

        charge = min(
            balance,
            max_charge,
            battery_capacity - soc
        )

        soc += charge * charge_efficiency

        system.iloc[i,
                    system.columns.get_loc('battery_flow')] = charge

        system.iloc[i,
                    system.columns.get_loc('curtailment')] = (
                        balance - charge
                    )

    # deficit
    else:

        discharge = min(
            abs(balance),
            max_discharge,
            soc
        )

        soc -= discharge

        system.iloc[i,
                    system.columns.get_loc('battery_flow')] = -discharge

        remaining = abs(balance) - discharge

        system.iloc[i,
                    system.columns.get_loc('grid_import')] = remaining

    system.iloc[i,
                system.columns.get_loc('battery_soc')] = soc

In [9]:
print(system[['battery_soc',
              'grid_import',
              'curtailment']].describe())

        battery_soc   grid_import  curtailment
count  17534.000000  17534.000000      17534.0
mean     297.721752  24904.049276          0.0
std     2631.726704  11620.623271          0.0
min        0.000000      0.000000          0.0
25%        0.000000  17199.750000          0.0
50%        0.000000  25636.500000          0.0
75%        0.000000  33136.500000          0.0
max    59326.550000  58588.000000          0.0


In [10]:
print(
    "Total grid imports:",
    system['grid_import'].sum()
)

print(
    "Hours with grid support:",
    (system['grid_import'] > 0).sum()
)

print(
    "Renewable curtailment:",
    system['curtailment'].sum()
)

Total grid imports: 436667600.00000006
Hours with grid support: 17080
Renewable curtailment: 0.0


In [11]:
#scenarios analysis
scenario = system.copy()

scenario['wind'] *= 2
scenario['solar'] *= 2

scenario['renewables'] = (
    scenario['wind']
    + scenario['solar']
)

scenario['net_balance'] = (
    scenario['renewables']
    - scenario['load']
)

In [12]:
#Simulation
system['battery_soc'] = 0.0
system['battery_flow'] = 0.0
system['grid_import'] = 0.0
system['curtailment'] = 0.0

soc = initial_soc

for i in range(len(system)):

    balance = system.iloc[i]['net_balance']

    # surplus renewables
    if balance > 0:

        charge = min(
            balance,
            max_charge,
            battery_capacity - soc
        )

        soc += charge * charge_efficiency

        system.iloc[i,
                    system.columns.get_loc('battery_flow')] = charge

        system.iloc[i,
                    system.columns.get_loc('curtailment')] = (
                        balance - charge
                    )

    # deficit
    else:

        discharge = min(
            abs(balance),
            max_discharge,
            soc
        )

        soc -= discharge

        system.iloc[i,
                    system.columns.get_loc('battery_flow')] = -discharge

        remaining = abs(balance) - discharge

        system.iloc[i,
                    system.columns.get_loc('grid_import')] = remaining

    system.iloc[i,
                system.columns.get_loc('battery_soc')] = soc